In [2]:
from langchain_neo4j import Neo4jGraph
import os
from dotenv import load_dotenv
load_dotenv(".env", override=True)

graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database="my-graphrag-db"
)

print("Connected to Neo4j")

Connected to Neo4j


In [3]:
from langchain_groq import ChatGroq

groq_api_key = os.environ.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=groq_api_key,
    reasoning_effort="low",
    model_kwargs={
        "parallel_tool_calls": False
    }
)

In [4]:
from langchain_core.documents import Document

movie_texts = [
    "Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.",
    "Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.",
    "Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008."
]
documents = [Document(page_content=text) for text in movie_texts]
documents

[Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.'),
 Document(metadata={}, page_content='Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.'),
 Document(metadata={}, page_content='Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.')]

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)
chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:", chunk.page_content)

Chunk 1: Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.
Chunk 2: Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.
Chunk 3: Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.


In [8]:
from langchain_neo4j import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Movie", "Genre"],
    allowed_relationships=["DIRECTED", "ACTED_IN", "IN_GENRE"]
)

graph_documents = llm_transformer.convert_to_graph_documents(chunks)
graph_documents

[GraphDocument(nodes=[Node(id='Christopher Nolan', type='Person', properties={}), Node(id='Leonardo Dicaprio', type='Person', properties={}), Node(id='Inception', type='Movie', properties={}), Node(id='Science Fiction', type='Genre', properties={})], relationships=[Relationship(source=Node(id='Christopher Nolan', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='DIRECTED', properties={}), Relationship(source=Node(id='Leonardo Dicaprio', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='ACTED_IN', properties={}), Relationship(source=Node(id='Inception', type='Movie', properties={}), target=Node(id='Science Fiction', type='Genre', properties={}), type='IN_GENRE', properties={})], source=Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.')),
 GraphDocument(nodes=[Node(id='Christopher No

In [7]:
# Inspect extracted graph data
for i, graph_doc in enumerate(graph_documents):
    print(f"\n--- Graph Document {i+1} ---")
    print("Nodes:")
    for node in graph_doc.nodes:
        print(node)
    print("Relationships:")
    for rel in graph_doc.relationships:
        print(rel)


--- Graph Document 1 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Leonardo Dicaprio' type='Person' properties={}
id='Inception' type='Movie' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='DIRECTED' properties={}
source=Node(id='Leonardo Dicaprio', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='ACTED_IN' properties={}
source=Node(id='Inception', type='Movie', properties={}) target=Node(id='Science Fiction', type='Genre', properties={}) type='IN_GENRE' properties={}

--- Graph Document 2 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Matthew Mcconaughey' type='Person' properties={}
id='Interstellar' type='Movie' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properti

In [9]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)
print("Graph data added to Neo4j")

Graph data added to Neo4j


In [10]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Document {id: STRING, text: STRING}
Person {id: STRING}
Movie {id: STRING}
Genre {id: STRING}
Relationship properties:

The relationships:
(:Document)-[:MENTIONS]->(:Person)
(:Document)-[:MENTIONS]->(:Movie)
(:Document)-[:MENTIONS]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


In [11]:
result = graph.query("""
MATCH (director:Person)-[:DIRECTED]->(movie:Movie)
OPTIONAL MATCH (actor:Person)-[:ACTED_IN]->(movie)
RETURN director.id AS director, movie.id AS movie, collect(actor.id) AS actors
""")

result

[{'director': 'Christopher Nolan',
  'movie': 'Inception',
  'actors': ['Leonardo Dicaprio']},
 {'director': 'Christopher Nolan',
  'movie': 'Interstellar',
  'actors': ['Matthew Mcconaughey']},
 {'director': 'Christopher Nolan',
  'movie': 'The Dark Knight',
  'actors': ['Christian Bale']}]

In [12]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)
response = chain.invoke({
    "query": "Which movies did Christopher Nolan direct?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: "Christopher Nolan"})-[:DIRECTED]->(m:Movie)
RETURN m
Full Context:
[{'m': {'id': 'Inception'}}, {'m': {'id': 'Interstellar'}}, {'m': {'id': 'The Dark Knight'}}]

> Finished chain.
Inception, Interstellar, The Dark Knight.


In [13]:
response = chain.invoke({
    "query": "Which movie did Christopher Nolan direct in 2010 and who acted in it?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: "Christopher Nolan"})-[:DIRECTED]->(m:Movie)
OPTIONAL MATCH (a:Person)-[:ACTED_IN]->(m)
RETURN m.id AS movieId, collect(a.id) AS actorsIds
Full Context:
[{'movieId': 'Inception', 'actorsIds': ['Leonardo Dicaprio']}, {'movieId': 'Interstellar', 'actorsIds': ['Matthew Mcconaughey']}, {'movieId': 'The Dark Knight', 'actorsIds': ['Christian Bale']}]

> Finished chain.
Inception was directed by Christopher Nolan in 2010, and Leonardo DiCaprio acted in it.
